In [1]:
import zipfile
import tempfile
import os
import shutil
import pandas as pd
from datetime import datetime

def extract_nested_zips(zip_path):
    """
    解压嵌套 zip 文件（zip 中还有 zip），
    返回所有最内层非 zip 文件的路径列表
    """

    extracted_files = []

    def _extract(zip_file_path, work_dir):
        with zipfile.ZipFile(zip_file_path, "r") as zf:
            zf.extractall(work_dir)

        for root, _, files in os.walk(work_dir):
            for file in files:
                file_path = os.path.join(root, file)

                if zipfile.is_zipfile(file_path):
                    # 如果还是 zip，继续解压
                    new_dir = tempfile.mkdtemp()
                    _extract(file_path, new_dir)
                else:
                    # 最内层文件
                    df = pd.read_csv(file_path)
                    df = df[df.SettlementPoint == 'LZ_HOUSTON']
                    df = df[['DeliveryDate', 'HourEnding', 'SettlementPointPrice']]
                    extracted_files.append(df)

    temp_root = tempfile.mkdtemp()

    try:
        _extract(zip_path, temp_root)
    finally:
        # 如果你希望后续还能使用文件路径，可注释掉
        pass
        # shutil.rmtree(temp_root)

    return extracted_files

/home/admin01/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
zip_file = "./raw_data/DAM Settlement Point Prices.zip"

files = extract_nested_zips(zip_file)

print(f"Total leaf files: {len(files)}")

Total leaf files: 4001


In [3]:
df = pd.concat(files)

In [4]:
df['DeliveryDate'] = pd.to_datetime(df.	DeliveryDate)

In [5]:
df = df.drop_duplicates()
df.columns = ['date', 'hour', 'dam_price']
df = df[df.date >= '2015-01-01 00:00:00']
df = df.sort_values(by=['date', 'hour'])
df['hour'] = df.hour.apply(lambda x: int(x[:2]))
df['date'] = df.date.apply(lambda x: x.strftime("%m/%d/%Y"))
df.index = range(len(df))
df

,date,hour,dam_price
0,01/01/2015,1,22.02
1,01/01/2015,2,21.44
2,01/01/2015,3,21.16
3,01/01/2015,4,20.94
4,01/01/2015,5,21.37
...,...,...,...
95923,12/10/2025,20,60.94
95924,12/10/2025,21,64.34
95925,12/10/2025,22,58.20
95926,12/10/2025,23,50.68


In [6]:
df.to_csv("./processed_data/dam_houston_price.csv", index=None)